# Métricas dos modelos

Este notebook lê `reports/model_metrics.json`, compara os modelos e mostra F2-score, F1-score, accuracy (acurácia), precision (precisão), recall, AUPRC e ROC AUC no holdout. A matriz de confusão corresponde ao modelo selecionado no treino.

Execute o treinamento com `python -m src.models.train` antes de abrir este notebook para atualizar os resultados.

In [ ]:
import json
from pathlib import Path
import pandas as pd
from IPython.display import display, Image

ROOT = Path.cwd()
if not (ROOT / 'reports').exists():
    ROOT = ROOT.parent
metrics_path = ROOT / 'reports' / 'model_metrics.json'
results = json.loads(metrics_path.read_text(encoding='utf-8'))
df = pd.DataFrame(results)
df.sort_values('cv_average_precision_mean', ascending=False, inplace=True)
columns = ['model', 'sampler', 'threshold', 'cv_average_precision_mean', 'cv_average_precision_std', 'cv_roc_auc_mean', 'cv_recall_mean', 'cv_f2_mean', 'test_average_precision', 'test_roc_auc', 'test_f2', 'test_f1', 'test_accuracy', 'test_precision', 'test_recall']
display(df[columns].round(3))

## Como interpretar

- **F1-score:** objetivo usado para escolher o threshold; equilibra precisão e recall.
- **F1-score:** equilíbrio entre precisão e recall.
- **Accuracy / acurácia:** fração total de classificações corretas; pode parecer alta devido ao desbalanceamento.
- **Precision / precisão:** proporção dos alertas previstos que são positivos.
- **AUPRC:** área resumida da curva precisão-recall; é especialmente útil para a classe AVC, que é rara.
- **Threshold:** ponto de corte estimado com previsões out-of-fold no conjunto de treino, maximizando F1.

Na matriz abaixo, linhas são rótulos reais e colunas são previsões. O quadro ao lado contém F2/F1-score, accuracy, precision e recall do modelo selecionado.

![Matriz de confusão](../reports/figures/confusion_matrix.png)

In [ ]:
display(Image(filename=str(ROOT / 'reports' / 'figures' / 'confusion_matrix.png')))

## Técnicas comparadas nesta execução

Foram comparados pesos de classe, Balanced Random Forest e, com o mesmo XGBoost, SMOTENC e oversampling aleatório nas proporções 10%, 30% e 50% da classe majoritária. A reamostragem ocorre somente dentro de cada fold.

Pelo critério configurado de AUPRC média em CV, o modelo continua sendo **XGBoost + RandomOverSampler 50%**. O threshold foi recalibrado para maximizar F1 usando somente previsões out-of-fold do treino: **0,2306**.

Nesse threshold, o holdout ficou com precisão **0,181**, recall **0,540**, F1 **0,271** e F2 **0,387**; identificou **27/50 casos positivos**, com 23 falsos negativos e 122 falsos positivos. Em CV, precisão foi `0,171`, recall `0,553` e F1 `0,261`. A busca OOF não encontrou threshold que levasse simultaneamente precisão, recall e F1 a `0,80`; não é correto escolher esse ponto olhando o holdout.

O Balanced Random Forest teve o maior F2 médio em CV (`0,422`) e recall de holdout `0,82` (41/50), mas o threshold salvo permanece associado ao modelo selecionado pela AUPRC. A tabela e a matriz abaixo mostram os resultados atuais.


In [ ]:
card = json.loads((ROOT / 'models' / 'model_card.json').read_text(encoding='utf-8'))
best = next(row for row in results if row['model'] == card['selected_model'] and row['sampler'] == card['sampler'])
selected_metrics = pd.Series({
    'modelo': best['model'], 'amostragem': best['sampler'],
    'threshold': best['threshold'], 'F2-score': best['test_f2'], 'F1-score': best['test_f1'],
    'Accuracy': best['test_accuracy'], 'Precision': best['test_precision'],
    'Recall AVC positivo': best['test_recall'], 'AUPRC': best['test_average_precision'],
    'ROC AUC': best['test_roc_auc'],
    'CV precision (threshold)': card['threshold_cv_metrics']['precision'],
    'CV recall (threshold)': card['threshold_cv_metrics']['recall'],
    'CV F1 (threshold)': card['threshold_cv_metrics']['f1']
}, name='valor').to_frame()
display(selected_metrics.round(3))
tn, fp = best['confusion_matrix'][0]
fn, tp = best['confusion_matrix'][1]
display(pd.DataFrame({'previsto 0': [tn, fn], 'previsto 1': [fp, tp]}, index=['real 0', 'real 1']))
print(f"AVC positivos detectados: {tp}/{tp + fn}; falsos negativos: {fn}; falsos positivos: {fp}.")
best_cv_f2 = max(results, key=lambda row: row['cv_f2_mean'])
print(f"Maior F2 médio em CV: {best_cv_f2['model']} + {best_cv_f2['sampler']} (CV F2={best_cv_f2['cv_f2_mean']:.3f}; holdout recall={best_cv_f2['test_recall']:.3f}; holdout F2={best_cv_f2['test_f2']:.3f}).")